# HUẤN LUYỆN THỬ NGHIỆM EXP-M1-DiffMul (CodeBERT — Forward tương tác [|hA - hB|, hA ⊙ hB] (LR=1e-5, CosineLR, Epochs=8, WD=0.05, Dropout=0.2))

Notebook này thực hiện huấn luyện kiến trúc tương tác vector mới **`[|hA - hB|, hA ⊙ hB]`** cho **M1** (CodeBERT):
- **Mã thử nghiệm:** `EXP-M1-DiffMul`
- **Mô hình cơ sở:** `M1` (CodeBERT)
- **Cấu hình tốt nhất kế thừa:** LR/Dropout/Weight Decay tối ưu từ các vòng tuning trước
- **Cơ chế tương tác mới:** Thay `[hA, hB, |hA - hB|]` (3d) bằng `[|hA - hB|, hA ⊙ hB]` (2d)
- **File cấu hình:** `configs/tuning/exp_m1_diff_mul.yaml`

### Cơ chế An toàn & Tự động dọn dẹp:
- Tự động dò tìm dataset BigCloneBench trong `/kaggle/input`.
- Chạy training kèm Early Stopping (patience=4) và logging chi tiết.
- Đánh giá chuẩn xác trên 5.000 cặp BigCloneBench Java test balanced.
- **Auto-Cleanup:** Tự động xóa sạch mã nguồn tạm, chỉ xuất ra checkpoint `.pt`, file đánh giá `.json`, `.csv` và file nén kết quả.

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ STREAM LOG AN TOÀN
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, yaml
from pathlib import Path

def sh(cmd, **kw):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path("/kaggle/working/live_progress.log")
    with log_file.open("a", encoding="utf-8") as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={rc}): {cmd}")
    return ""

print("Python:", sys.version.split()[0], flush=True)
print("Thư mục làm việc hiện tại:", os.getcwd(), flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN VÀO THƯ MỤC RIÊNG
payload = '...'
raw_bytes = base64.b64decode(payload)
repo_root = Path('/kaggle/working/project')
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(raw_bytes), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Thu muc lam viec da chuyen sang:', os.getcwd())
sh('ls -la')

In [ ]:
# 3. KIỂM TRA PHẦN CỨNG GPU & DEPENDENCIES
import torch
print('CUDA kha dung:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Ten GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    raise SystemError('Kaggle kernel chua bat GPU accelerator! Vui long bat GPU T4 x2 hoac P100.')

sh('pip install -q -U transformers datasets scikit-learn pyarrow \"tree-sitter>=0.23.0\" \"tree-sitter-java>=0.23.5\"')

In [ ]:
# 4. TỰ ĐỘNG DÒ TÌM DATASET BIGCLONEBENCH
input_root = Path('/kaggle/input')
ds_root = None
if input_root.exists():
    for p in input_root.iterdir():
        if p.is_dir() and ('semanticcodeclone' in p.name.lower() or 'bigclonebench' in p.name.lower()):
            ds_root = p
            break
print(f'Thu muc dataset phat hien: {ds_root}')
if ds_root:
    sh(f'ls -la {ds_root}/')

In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU CHUẨN (TRAIN 30k, VAL 3k, TEST 5k CÂN BẰNG 50:50)
data_raw = Path('data/raw/bigclonebench')
data_raw.mkdir(parents=True, exist_ok=True)

cached_parquet = ds_root / 'bcb_train.parquet' if ds_root else None
if cached_parquet and cached_parquet.exists():
    print('Phat hien du lieu parquet san co tu dataset, sao chep truc tiep...')
    sh(f'cp {ds_root}/*.parquet data/raw/bigclonebench/')
elif (data_raw / 'bcb_train.parquet').exists() and (data_raw / 'bcb_test.parquet').exists():
    print('Cac file parquet da ton tai day du, bo qua download!')
else:
    print('Tai truc tiep cac split BigCloneBench tu Hugging Face...')
    sh('python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000')

sh('ls -lh data/raw/bigclonebench/*.parquet')
assert (data_raw / 'bcb_train.parquet').exists(), 'bcb_train.parquet phai ton tai truoc khi train!'
assert (data_raw / 'bcb_test.parquet').exists(), 'bcb_test.parquet phai ton tai truoc khi train!'

In [ ]:
# 6. HUẤN LUYỆN & ĐÁNH GIÁ: EXP-M1-DiffMul (Forward tương tác [|hA - hB|, hA ⊙ hB] (LR=1e-5, CosineLR, Epochs=8, WD=0.05, Dropout=0.2))
print('=' * 70)
print('>>> BẮT ĐẦU HUẤN LUYỆN: EXP-M1-DiffMul (Forward tương tác [|hA - hB|, hA ⊙ hB] (LR=1e-5, CosineLR, Epochs=8, WD=0.05, Dropout=0.2)) <<<')
print('=' * 70)
cfg = 'configs/tuning/exp_m1_diff_mul.yaml'
t_start = time.perf_counter()

# Preflight check
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode preflight --device cuda')

# Train (Early stopping monitor val_loss, time budget 5h)
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode train --device cuda --time-budget-s 18000')

# Evaluate on 5k test balanced
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode evaluate --device cuda')

print(f'>>> HOÀN TẤT EXP-M1-DiffMul TRONG {(time.perf_counter() - t_start)/60:.1f} PHÚT! <<<')

In [ ]:
# 7. THU THẬP KẾT QUẢ & AUTO-CLEANUP
print('=' * 70)
print('THU THẬP KẾT QUẢ VÀ DỌN DẸP SẠCH MÃ NGUỒN...')
print('=' * 70)

WORKING_DIR = Path('/kaggle/working')
CLEAN_DIR = WORKING_DIR / 'clean_results'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

results_dir = Path('/kaggle/working/project/results')
copied_files = []
if results_dir.exists():
    for p in results_dir.rglob('*'):
        if p.is_file() and p.suffix in ('.pt', '.json', '.csv'):
            subfolder = p.parent.name
            out_name = f"{subfolder}_{p.name}" if not p.name.startswith(subfolder) else p.name
            dest_clean = CLEAN_DIR / out_name
            shutil.copy2(p, dest_clean)
            shutil.copy2(p, WORKING_DIR / out_name)
            copied_files.append(out_name)

if (WORKING_DIR / 'live_progress.log').exists():
    shutil.copy2(WORKING_DIR / 'live_progress.log', CLEAN_DIR / 'live_progress.log')

tar_name = 'm1_diff_mul_clean_bundle.tar.gz'
tar_path = WORKING_DIR / tar_name
with tarfile.open(tar_path, 'w:gz') as tar:
    for f in CLEAN_DIR.glob('*'):
        tar.add(f, arcname=f.name)
print(f'>>> Da tao file goi ket qua: {tar_name} ({tar_path.stat().st_size / 1024 / 1024:.2f} MB) <<<')

os.chdir('/kaggle/working')
junk_paths = [
    Path('/kaggle/working/project'),
    Path('/kaggle/working/data'),
    Path('/kaggle/working/artifacts'),
    Path('/kaggle/working/clean_results'),
]
for jp in junk_paths:
    if jp.exists():
        shutil.rmtree(jp, ignore_errors=True)
        print(f'Da xoa sach thu muc: {jp.name}/')

print('\n' + '=' * 70)
print('DANH SÁCH FILE ĐẦU RA SẠCH DUY NHẤT CÒN LẠI TRÊN /kaggle/working/ :')
print('=' * 70)
total_size = 0
for item in sorted(WORKING_DIR.glob('*')):
    if item.is_file():
        sz = item.stat().st_size
        total_size += sz
        sz_str = f'{sz/1024/1024:.2f} MB' if sz > 1024*1024 else f'{sz/1024:.1f} KB'
        print(f'  - {item.name:<45} | Size: {sz_str}')
print(f'TONG DUNG LUONG DAU RA: {total_size/1024/1024:.2f} MB')
print('>>> QUÁ TRÌNH HUẤN LUYỆN VÀ DỌN DẸP ĐÃ HOÀN TẤT AN TOÀN 100%! <<<')